In [1]:
__author__ = "Swaminathan Sundar", "Rahul Kakodkar", "Efstratios Pistikopoulos"
__copyright__ = "Copyright 2022, Multi-parametric Optimization & Control Lab"
__credits__ = ["Swaminathan Sundar", "Rahul Kakodkar",
               "Efstratios N. Pistikopoulos"]
__license__ = "Open"
__version__ = "1.0.0"
__maintainer__ = "Swaminathan Sundar"
__email__ = "swaminathan.sundar@tamu.edu"
__status__ = "Production"

# from pyomo.environ import *
# from pyomo.dae import *
# import matplotlib.pyplot as plt
# import numpy as np
# import pandas as pd
# from pandas import DataFrame
# import time
# from matplotlib import rc
# # from idaes.core.util.model_statistics import degrees_of_freedom
# import os
# from src.components.state import State
# from src.components.task import Task
# from src.components.unit import Unit
# # import olca


In [2]:
import pandas 
from numpy import poly1d, polyfit, arange
from src.energiapy.components.temporal_scale import Temporal_scale
from src.energiapy.components.resource import Resource
from src.energiapy.components.process import Process
from src.energiapy.components.material import Material
from src.energiapy.components.location import Location
from src.energiapy.components.network import Network
from src.energiapy.components.scenario import Scenario
from src.energiapy.components.transport import Transport
from src.energiapy.components.result import Result 
from src.energiapy.model.formulate_milp import formulate_milp
from src.energiapy.utils.data_utils import get_data, make_henry_price_df
from src.energiapy.utils.nsrdb_utils import fetch_nsrdb_data
from src.energiapy.plot import plot
from src.energiapy.model.pyomo_solve import solve
from src.energiapy.utils.cluster_utils import reduce_scenario, agg_hierarchial_elbow,\
    Clustermethod, dynamic_warping, dynamic_warping_matrix, dynamic_warping_path
from src.energiapy.utils.data_utils import load_results
import matplotlib.pyplot as plt
from matplotlib import rc
from itertools import product

In [3]:
SkimMilk = Resource(name = 'SkimMilk', price = 0.3, block = 'Input')
PastMilk = Resource(name = 'PasteurizedMilk', price = 0, block = 'Intermediate')
Culture = Resource(name = 'Culture', price = 0, block = 'Input')
Whey = Resource(name = 'Whey', price = 0, block = 'Intermediate')
Curd = Resource(name = 'Curd', price = 0, block = 'Intermediate')
SolCurd = Resource(name = 'SolCurd', price = 0, block = 'Intermediate')
Waste_1 = Resource(name = 'Waste1', price = 0, block = 'Waste')
Cream = Resource(name = 'Cream', price = 1.5, block = 'Input')
Cheese = Resource(name = 'Cheese', price = 0, block = 'Intermediate')
StdBld9 = Resource(name = 'StdBld9', revenue= 6.36, sell = True, demand = True, block = 'Output')
StdBld10 = Resource(name = 'StdBld10', revenue= 6.36, sell = True, demand = True, block = 'Output')
Protein = Resource(name = 'Protein', revenue= 0.2, sell = True, block = 'ByProduct')
Permeate = Resource(name = 'Permeate', price = 0, sell = True, block = 'Intermediate')
Lactose = Resource(name = 'Lactose', revenue= 0.2, block = 'ByProduct')
Waste_2 = Resource(name = 'Waste2', revenue= 0, block = 'Waste')
Steam = Resource(name = 'Steam', price = 0.011, block = 'Utility')

In [4]:
Skim2 = Resource(SkimMilk.__dict__)

In [5]:
#Tasks

Pasteurizer = Process(name = 'Pasteurizer', prod_min = 50, prod_max = 2700, conversion = {SkimMilk: -1, PastMilk: 1}, \
    cost= {'CAPEX': 0, 'Fixed O&M': 50, 'Variable O&M': 0.4, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 1)
Vat1 = Process(name = 'Vat1', prod_min = 50, prod_max = 600, conversion = {Culture: -0.12, PastMilk: -0.88, Whey: 0.896, Curd: 0.104}, \
    cost= {'CAPEX': 0, 'Fixed O&M': 75, 'Variable O&M': 0.45, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 8)
Vat2 = Process(name = 'Vat2', prod_min = 50, prod_max = 800, conversion = {Culture: -0.12, PastMilk: -0.88, Whey: 0.896, Curd: 0.104},\
    cost= {'CAPEX': 0, 'Fixed O&M': 81, 'Variable O&M': 0.5, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 8)
Vat3 = Process(name = 'Vat3', prod_min = 50, prod_max = 1940, conversion = {Culture: -0.12, PastMilk: -0.88, Whey: 0.896, Curd: 0.104},\
    cost= {'CAPEX': 0, 'Fixed O&M': 89, 'Variable O&M': 0.55, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 8)
Drainer = Process(name = 'Drainer', prod_min = 5, prod_max = 225, conversion = {Curd: -1, SolCurd: 0.9, Waste_1: 0.1},\
    cost= {'CAPEX': 0, 'Fixed O&M': 45, 'Variable O&M': 0.3, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 1)
Blender_1 = Process(name = 'Blender1', prod_min = 5, prod_max = 300, conversion = {Cream: -0.33, SolCurd: -0.67, Cheese: 1},\
    cost= {'CAPEX': 0, 'Fixed O&M': 30, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 2)
Blender_2 = Process(name = 'Blender2', prod_min = 5, prod_max = 300, conversion = {Cream: -0.33, SolCurd: -0.67, Cheese: 1},\
    cost= {'CAPEX': 0, 'Fixed O&M': 30, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 2)
Filler_9 = Process(name = 'Filler9', prod_min = 5, prod_max = 150, conversion = {Cheese: -1, StdBld9: 1},\
    cost= {'CAPEX': 0, 'Fixed O&M': 25, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 1)
Filler_10 = Process(name = 'Filler10', prod_min = 5, prod_max = 150, conversion = {Cheese: -1, StdBld10: 1}, \
    cost= {'CAPEX': 0, 'Fixed O&M': 25, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 1)
UltraFilMem = Process(name = 'UFMem', prod_min = 50, prod_max = 5300, conversion = {Whey: -1, Protein: 0.03 , Permeate: 0.97},\
    cost= {'CAPEX': 0, 'Fixed O&M': 45, 'Variable O&M': 0.3, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 8, label = 'Ultra_Filtratiion_Membrane')
RevOsmoMem = Process(name = 'ROMem', prod_min = 50, prod_max = 5300, conversion = {Permeate: -1, Lactose: 0.9 , Waste_2: 0.1},\
    cost= {'CAPEX': 0, 'Fixed O&M': 120, 'Variable O&M': 0.4, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 2, label = 'Reverse_Osmosis_Membrane')


In [6]:

Milk_Silo = Process(name = 'MilkSilo', prod_min = 0, prod_max = 14100,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = SkimMilk, block = 'Storage')
Culture_Silo = Process(name = 'CultureSilo', prod_min = 0, prod_max = 5000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Culture, block = 'Storage')
Curd_Tank_1 = Process(name = 'CurdTank1', prod_min = 0, prod_max = 2000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Curd, block = 'Storage')
Curd_Tank_2 = Process(name = 'CurdTank2', prod_min = 0, prod_max = 2000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Curd, block = 'Storage')
Curd_Tank_3 = Process(name = 'CurdTank3', prod_min = 0, prod_max = 2000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Curd, block = 'Storage')
Waste_Tank_1 = Process(name = 'WasteTank1', prod_min = 0, prod_max = 2000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Waste_1, block = 'Storage')
Waste_Tank_2 = Process(name = 'WasteTank2', prod_min = 0, prod_max = 2000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Waste_2, block = 'Storage')
Cream_Silo = Process(name = 'CreamSilo', prod_min = 0, prod_max = 2800,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Cream, block = 'Storage')
Warehouse_9 = Process(name = 'Warehouse9', prod_min = 0, prod_max = 1000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = StdBld9, block = 'Storage')
Warehouse_10 = Process(name = 'Warehouse10', prod_min = 0, prod_max = 1000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = StdBld10, block = 'Storage')
Protein_Tank = Process(name = 'ProteinTank', prod_min = 0, prod_max = 3000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Protein, block = 'Storage')
Lactose_Tank = Process(name = 'LactoseTank', prod_min = 0, prod_max = 3000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Lactose, block = 'Storage')

In [7]:
Milk_Silo.conversion, Milk_Silo.conversion_discharge


({SkimMilk: -1, SkimMilk_stored: 1}, {SkimMilk_stored: -1, SkimMilk: 1})

In [8]:
scale = Temporal_scale([5508*2])# annual working hours, each discretization represents 30 mins

In [9]:

Horizon = 5508*2
Cycle_time = 12*2
CCF = 1
demand = 1500
alpha = 1
beta = 1
If = 1.01
CCF = 1
Hp  = Horizon


#%%
#Sets
Processes = [Pasteurizer, Vat1, Vat2, Vat3, Drainer, Blender_1, Blender_2, Filler_9, Filler_10, UltraFilMem, RevOsmoMem, \
    Milk_Silo, Culture_Silo, Curd_Tank_1, Curd_Tank_2, Curd_Tank_3, Waste_Tank_1, Waste_Tank_2, Cream_Silo, Warehouse_9, Warehouse_10, Protein_Tank, Lactose_Tank]

Resources = [Steam, Culture, SkimMilk, PastMilk, Whey, Curd, SolCurd, Waste_1, Cream, Cheese, StdBld9, StdBld10, Protein, Permeate, Lactose, Waste_2, Steam]
Time = range(0, Horizon+1)


process_list = [i.name for i in Processes]
resource_list = [i.name for i in Resources]

T = range(0, Cycle_time + 1)

# I = [i.name for i in Tasks]
# J = [i.name for i in Units]
# S = [i.name for i in States]
# U = [Steam.name]
# WS = [Waste_1.name, Waste_2.name]
# T = range(0, Cycle_time + 1)

# TK = [Milk_Silo, Culture_Silo, Curd_Tank_1, Curd_Tank_2, Curd_Tank_3, Waste_Tank_1, Waste_Tank_2, Cream_Silo, Warehouse_9, Warehouse_10, Protein_Tank, Lactose_Tank]


In [ ]:


# Ij = {j: j.__dict__['tasks'] for j in Units if j.__dict__['tasks'] is not None}
Ki = {i: i.__dict__['units'] for i in Tasks}                                    # K[i] set of units capable of task i

Ij = {}
for k,v in Ki.items():
    for x in v:
        Ij.setdefault(x,[]).append(k)

Si = {i: i.__dict__['input'] for i in I}
Sbari = {i: i.__dict__['output'] for i in I}
Ts = {}
for k,v in Si.items():
    for x in v:
         Ts.setdefault(x,[]).append(k)
Tbars = {}
for k,v in Sbari.items():
    for x in v:
         Tbars.setdefault(x,[]).append(k)
# print(Tbars, Ts)
Ks = {s: s.__dict__['storage'] for s in S if s.__dict__['storage'] is not None}

# print(Ks)
P = {(i,s): i.__dict__['output_time'][s] for i in I for s in i.__dict__['output']}          # P[(i,s)] time for task i output to state s  
pi = {i: max([P[(i,s)] for s in Sbari[i]]) for i in I}           # p[i] completion time for task i
print(P)
rho = {(i,s): i.__dict__['input_fraction'][s] for i in Tasks for s in i.__dict__['input']}        # rho[(i,s)] input fraction of task i from state s
rho_ = {(i,s): i.__dict__['output_fraction'][s] for i in Tasks for s in i.__dict__['output']}       # rho_[(i,s)] output fraction of task i to state s

print(rho, rho_)

model = ConcreteModel()

#Sets
model.I = Set(initialize = [i.name for i in I])
model.J = Set(initialize = [i.name for i in J])
model.S = Set(initialize = [i.name for i in S])
model.U = Set(initialize = [i.name for i in U])
model.WS = Set(initialize = [i.name for i in WS])
model.T = Set(initialize = T)
model.TK = Set(initialize = [i.name for i in TK])

# model.I = Set(initialize = I)
# model.J = Set(initialize = J)
# model.S = Set(initialize = S)
# model.U = Set(initialize = U)
# model.WS = Set(initialize = WS)
# model.T = Set(initialize = T)
# model.TK = Set(initialize = TK)


#Parameters
Ds = 137700/2
Ss0 = {s: s.__dict__['initial_value'] for s in S}

# #Variables
# model.Ej = Var(model.J, domain = Binary)
# model.Etk = Var(model.TK, domain = Binary)
# model.W = Var(model.I, model.J, model.T, domain = Binary)

model.Ej = Var(model.J, domain = NonNegativeReals)
model.Etk = Var(model.TK, domain = NonNegativeReals)
model.W = Var(model.I, model.J, model.T, domain = NonNegativeReals)

model.Vj = Var(model.J, domain = NonNegativeReals)
model.Vtk = Var(model.TK, domain = NonNegativeReals)
model.B = Var(model.I, model.J, model.T, domain = NonNegativeReals)
model.St = Var(model.S, model.T, domain = NonNegativeReals)
model.Waste = Var(model.WS, model.T, domain = NonNegativeReals)
model.Util = Var(model.U, model.T, domain = NonNegativeReals)
model.setup_cost = Var(domain = NonNegativeReals)
model.revenue = Var(domain = NonNegativeReals)

initial = [Culture.name, SkimMilk.name, Cream.name]

for s in model.S:
    if s not in initial:
        # print(s)
        model.St[s, 0].fix(0)
        
for s in model.S:
    if s not in [i.name for i in Ks]:     
        model.St[s,:].fix(0) 


# for s in initial:
#     model.St[s,0].fix(10000)


# from itertools import product 
# for i in J:
#     model.Ej[i].fix(0.2) 

# for i in TK:
#     model.Etk[i].fix(0.2)
    
# for i,j,k in product(I, J, T):
#     model.W[i,j,k].fix(0.2)


#Constraints

def unit_allocation(instance, unit, time):
    """Unit Allocation Constraints
    A unit can only be allocated to one task 
    Args:
        instance (_type_): _description_
        time (_type_): _description_
    """ 
    lhs = 0
    if unit in [i.name for i in Ij]:
        for i in Ij[unit]:
            if time  - pi[i] + 1 > 0:
                tdprime = time - pi[i] + 1
            else:
                tdprime = time - pi[i] + 1 + Cycle_time 
            for tprime in range(tdprime, time + 1):          
                lhs += instance.W[i,unit,tprime]
        expr = lhs <= instance.Ej[unit]
    else:
        expr = Constraint.Skip
    return expr


#%%
model.unit_allocation_cons = Constraint(model.J, model.T, rule = unit_allocation, doc = 'Unit Allocation Constraints')
#%%
def unit_capacity_lb(instance, task, unit):
    """Unit Capacity Lower Bound Constraints
        
    Args:
        instance (_type_): _description_
        time (_type_): _description_
    """ 
    if unit in Ki[task]:
        Vjmin = unit.__dict__['min_capacity']
        expr = Vjmin*instance.Ej[unit] <= instance.Vj[unit]
    else:
        expr = Constraint.Skip
    return expr

model.unit_capacity_lb_cons = Constraint(model.I, model.J, rule = unit_capacity_lb, doc = 'Unit Capacity LB Constraints')

def unit_capacity_ub(instance, task, unit):
    """Unit Capacity Upper Bound Constraints
        
    Args:
        instance (_type_): _description_
        time (_type_): _description_
    """ 
    if unit in Ki[task]:
        Vjmax = unit.__dict__['max_capacity']
        expr = instance.Vj[unit] <= Vjmax*instance.Ej[unit]
    else:
        expr = Constraint.Skip
    return expr

model.unit_capacity_ub_cons = Constraint(model.I, model.J, rule = unit_capacity_ub, doc = 'Unit Capacity UB Constraints')

def batch_size_lb(instance, task, unit, time):
    """Batch Size Production Lower Bound

    Args:
        instance (_type_): _description_
        task (_type_): _description_
        unit (_type_): _description_
        time (_type_): _description_
    """
    if unit in Ki[task]:
         Vjmin = unit.__dict__['min_capacity']
         expr = instance.W[task, unit, time]*Vjmin <= instance.B[task, unit, time]
    else:
        expr = Constraint.Skip
    return expr

model.batch_size_lb_cons = Constraint(model.I, model.J, model.T, rule = batch_size_lb, doc = 'Batch Size LB Constraints')

def batch_size_ub(instance, task, unit, time):
    """Batch Size Production Upper Bound

    Args:
        instance (_type_): _description_
        task (_type_): _description_
        unit (_type_): _description_
        time (_type_): _description_
    """
    if unit in Ki[task]:
         Vjmax = unit.__dict__['max_capacity']
         expr = instance.W[task, unit, time]*Vjmax <= instance.B[task, unit, time]
    else:
        expr = Constraint.Skip
    return expr

model.batch_size_ub_cons = Constraint(model.I, model.J, model.T, rule = batch_size_ub, doc = 'Batch Size UB Constraints')

def production_capacity(instance, task, unit, time):
    """Production constraint due to Unit Capacity

    Args:
        instance (_type_): _description_
        task (_type_): _description_
        unit (_type_): _description_
        time (_type_): _description_
    """
    if unit in Ki[task]:
        expr = instance.B[task, unit, time] <= instance.Vj[unit]
    else:
        expr = Constraint.Skip
    return expr

model.production_capacity_cons = Constraint(model.I, model.J, model.T, rule = production_capacity, doc = 'Production Capacity Constraints')

def storage_capacity_lb(instance, state, tanks, time):
    """Storage Upper Bound Constraints due to unit capacity

    Args:
        instance (_type_): _description_
        state (_type_): _description_
        tanks (_type_): _description_
        time (_type_): _description_
    """
    if state in Ks and tanks in Ks[state]:
        expr = 0 <= instance.St[state, time]
    else:
        expr = Constraint.Skip
    return expr

model.storage_capacity_lb_cons = Constraint(model.S, model.TK, model.T, rule = storage_capacity_lb, doc = 'Storage Capacity Upper Bound Constraints')

def storage_capacity_ub(instance, state, tanks, time):
    """Storage Upper Bound Constraints due to unit capacity

    Args:
        instance (_type_): _description_
        state (_type_): _description_
        tanks (_type_): _description_
        time (_type_): _description_
    """
    if state in Ks and tanks in Ks[state]:
        expr = instance.St[state, time] <= instance.Vtk[tanks]
    else:
        expr = Constraint.Skip
    return expr

model.storage_capacity_ub_cons = Constraint(model.S, model.TK, model.T, rule = storage_capacity_ub, doc = 'Storage Capacity Upper Bound Constraints')

def storage_lb(instance, state, tank):
    """Storage Lower Bound Constraints
        
    Args:
        instance (_type_): _description_
        time (_type_): _description_
    """ 
    if state in Ks and tank in Ks[state]:
        Vtkmin = tank.__dict__['min_capacity']
        expr = Vtkmin*instance.Etk[tank] <= instance.Vtk[tank]
    else:
        expr = Constraint.Skip
    return expr

model.storage_lb_cons = Constraint(model.S, model.TK, rule = storage_lb, doc = 'Storage LB Constraints')

def storage_ub(instance, state, tank):
    """Storage Upper Bound Constraints
        
    Args:
        instance (_type_): _description_
        time (_type_): _description_
    """ 
    if state in Ks and tank in Ks[state]:
        Vtkmax = tank.__dict__['max_capacity']
        expr = instance.Vtk[tank] <= Vtkmax*instance.Etk[tank]
    else:
        expr = Constraint.Skip
    return expr

model.storage_ub_cons = Constraint(model.S, model.TK, rule = storage_ub, doc = 'Storage UB Constraints')


def mass_balance(instance, state, time):
    """Mass Balance Constraints

    Args:
        instance (_type_): _description_
        state (_type_): _description_
        time (_type_): _description_
    """
    lhs = 0
    rhs = 0
    lhs = instance.St[state, time]
    if time != 0:
        rhs = instance.St[state, time -1] 
    if state in Ts:
        rhs -= sum(rho[task, state]*instance.B[task, unit, time] for task in Ts[state] for unit in Ki[task])
    if state in Tbars:
        for task in Tbars[state]:
            if time - P[task, state] >= 1:         
                rhs += sum(rho_[task, state]*instance.B[task, unit, time - P[task, state]] for unit in Ki[task]) 
            else:
                rhs += sum(rho_[task, state]*instance.B[task, unit, time - P[task, state] + Cycle_time] for unit in Ki[task])
    expr = lhs == rhs
    return expr


model.mass_balance_cons = Constraint(model.S, model.T, rule = mass_balance, doc = 'Mass Balance Constraints')

def production_requirement(instance, state):
    """Production Requirement Constraints

    Args:
        instance (_type_): _description_
    """
    if state is StdBld9 or state is StdBld10:
        expr = (instance.St[state, Cycle_time] - instance.St[state, 0])*Horizon <= Ds*Cycle_time
    else:
        expr = Constraint.Skip
    return expr

model.production_requirement_cons = Constraint(model.S, rule = production_requirement, doc = 'Production Requirement Constraints')

def utility(instance, utility, time):
    """Utility Constraints

    Args:
        instance (_type_): _description_
        utility (_type_): _description_
        time (_type_): _description_
    """
    return instance.Util[utility, time] == sum(alpha*instance.B[task, unit, time] for task in I for unit in Ki[task])

model.utility_cons = Constraint(model.U, model.T, rule = utility, doc = 'Utility Constraints')

def waste(instance, waste, time):
    """Waste Constraint

    Args:
        instance (_type_): _description_
        waste (_type_): _description_
        time (_type_): _description_
    """
    
    return instance.Waste[waste, time] == sum(beta*instance.B[task, unit, time] for task in I for unit in Ki[task])

model.waste_cons = Constraint(model.WS, model.T, rule = waste, doc = 'Waste Constraints')

def facility_cost(instance):
    """Cost of Facility

    Args:
        instance (_type_): _description_
    """
    
    return instance.setup_cost == (sum(instance.Ej[unit]*unit.__dict__['fixed_cost'] + instance.Vj[unit]*unit.__dict__['variable_cost'] for unit in instance.J) + sum(instance.Etk[tank]*tank.__dict__['fixed_cost'] + instance.Vtk[tank]*tank.__dict__['variable_cost'] for tank in instance.TK))*If*CCF 

model.facility_cost_cons = Constraint(rule = facility_cost, doc = 'Facility Cost Constraint')

def revenue_cost(instance):
    """Revenue

    Args:
        instance (_type_): _description_
    """
    return instance.revenue == (sum(state.__dict__['cost'] * (instance.St[state, Cycle_time] - instance.St[state, 0]) for state in instance.S) - sum(utility.__dict__['cost'] * instance.Util[utility, time] for utility in instance.U for time in instance.T))*Horizon/Cycle_time 

model.revenue_cost_cons = Constraint(rule = revenue_cost, doc = 'Revenue Constraint')

def objective(instance):
    """Objective Function to minimize annual cost

    Args:
        instance (_type_): _description_
    """
    return instance.revenue - instance.setup_cost

model.dual = Suffix(direction=Suffix.IMPORT_EXPORT)

        
model.Objective = Objective(rule = objective, sense = maximize, doc = 'Objective Function')

opt = SolverFactory('gurobi', solver_io = 'python')
results = opt.solve(model)
results.write()
#%%

model.pprint()
# def mass_balance(instance, state, time):
#     """Mass Balance Constraints

#     Args:
#         instance (_type_): _description_
#         state (_type_): _description_
#         time (_type_): _description_
#     """
#     print(state,time)
#     if state in Tbars:
#         # tprime = {}
#         # for i in Tbars[state]:
#         #     tprime.setdefault((i,state)).append(0)
#         #     # tprime[i].append(0)
#         #     if time - P[i, state] >= 1:
#         #         tprime[i, state] = time - P[i, state]
#         #     else:
#         #         tprime[i, state] = time - P[i, state] + Cycle_time
#         if time - P[task, state] >= 1:         
#             lhs = instance.St[state, time]
#             rhs = instance.St[state, time -1] - sum(rho(task, state)*instance.B[task, unit, time] for task in Ts for unit in Ki[task]) \
#                 + sum(rho_[task, state]*instance.B[task, unit, time - P[task, state]] for task in Tbars[state] for unit in Ki[task])
#             expr = lhs == rhs 
#         else:
#             lhs = instance.St[state, time]
#             rhs = instance.St[state, time -1] - sum(rho(task, state)*instance.B[task, unit, time] for task in Ts for unit in Ki[task]) \
#                 + sum(rho_[task, state]*instance.B[task, unit, time - P[task, state] + Cycle_time] for task in Tbars[state] for unit in Ki[task])
#             expr = lhs == rhs
#     else:
#         expr = Constraint.Skip
#     return expr
# %%
#Storage Inventory
pd.DataFrame([[model.St[s,t]() for s in model.S] for t in model.T], columns = model.S, index = model.T)

# %%
plt.figure(figsize=(10,10))
for (s,idx) in zip(model.S,range(0,len(model.S))):
    if s in Ks:
        plt.subplot(ceil(len(model.S)/3),3,idx+1)
        tlast,ylast = 0, s.__dict__['initial_value']
        for (t,y) in zip(list(model.T),[model.St[s,t]() for t in model.T]):
            plt.plot([tlast,t,t],[ylast,ylast,y],'b')
            #plt.plot([tlast,t],[ylast,y],'b.',ms=10)
            tlast,ylast = t,y
        plt.ylim(0,1.1*max(tank.__dict__['max_capacity'] for tank in Ks[s] ))
        plt.plot([0,Cycle_time],[max(tank.__dict__['max_capacity'] for tank in Ks[s] if s in Ks),max(tank.__dict__['max_capacity'] for tank in Ks[s] if s in Ks)],'r--')
        plt.title(s.__dict__['name'])
plt.tight_layout()
# %%


In [ ]:


#Storages


#Tasks
Pasteurize = Task(name = 'Pasteurize', time = 1, input = [SkimMilk], output = [PastMilk], input_time = {SkimMilk: 0}, output_time = {PastMilk: 1}, input_fraction = {SkimMilk: 1}, output_fraction = {PastMilk: 30}, units = [Pasteurizer])
VatProc = Task(name = 'Vat Proc', time = 8, input = [Culture, PastMilk], output = [Whey, Curd], input_time = {Culture: 0, PastMilk: 0}, output_time = {Whey: 8, Curd: 8}, input_fraction = {Culture: 0.12, PastMilk: 0.88}, output_fraction = {Whey: 0.896, Curd: 0.104}, units = [Vat1, Vat2, Vat3])
Drain = Task(name = 'Drain', time = 1, input = [Curd], output = [SolCurd, Waste_1], input_time = {Curd: 0}, output_time = {SolCurd: 1, Waste_1: 1}, input_fraction = {Curd: 1}, output_fraction = {SolCurd: 0.9, Waste_1: 0.1}, units = [Drainer])
Blend = Task(name = 'Blend', time = 2, input = [Cream, SolCurd], output = [Cheese], input_time = {Cream: 0, SolCurd: 0}, output_time = {Cheese: 2}, input_fraction = {Cream: 0.33, SolCurd: 0.67}, output_fraction = {Cheese: 1}, units = [Blender_1, Blender_2])
Fill_9 = Task(name = 'Fill 9', time = 1, input = [Cheese], output = [StdBld9], input_time = {Cheese: 0}, output_time = {StdBld9: 1}, input_fraction = {Cheese: 1}, output_fraction = {StdBld9: 1}, units = [Filler_9])
Fill_10 = Task(name = 'Fill 10', time = 1, input = [Cheese], output = [StdBld10], input_time = {Cheese: 0}, output_time = {StdBld10: 1}, input_fraction = {Cheese: 1}, output_fraction = {StdBld10: 1}, units = [Filler_10])
UltraFiltr = Task(name = 'Ultra Filtration', time = 8, input = [Whey], output = [Protein, Permeate], input_time = {Whey: 0}, output_time = {Protein: 8, Permeate: 8}, input_fraction = {Whey: 1}, output_fraction = {Protein: 0.03 , Permeate: 0.97}, units = [UltraFilMem])
RevOsmos = Task(name = 'Reverse Osmosis', time = 2, input = [Permeate], output = [Lactose, Waste_2], input_time = {Permeate: 0}, output_time = {Lactose: 2, Waste_2: 2}, input_fraction = {Permeate: 1}, output_fraction = {Lactose: 0.9 , Waste_2: 0.1}, units = [RevOsmoMem])


Horizon = 5508*2
Cycle_time = 12*2
CCF = 1
demand = 1500
alpha = 1
beta = 1
If = 1.01
CCF = 1
Hp  = Horizon


#%%
#Sets
Tasks = [VatProc, Pasteurize, Drain, Blend, Fill_9, Fill_10, UltraFiltr, RevOsmos]
Units = [Pasteurizer, Vat1, Vat2, Vat3, Drainer, Blender_1, Blender_2, Filler_9, Filler_10, UltraFilMem, RevOsmoMem, \
    Milk_Silo, Culture_Silo, Curd_Tank_1, Curd_Tank_2, Curd_Tank_3, Waste_Tank_1, Waste_Tank_2, Cream_Silo, Warehouse_9, Warehouse_10, Protein_Tank, Lactose_Tank]
States = [Culture, SkimMilk, PastMilk, Whey, Curd, SolCurd, Waste_1, Cream, Cheese, StdBld9, StdBld10, Protein, Permeate, Lactose, Waste_2, Steam]
Time = range(0, Horizon+1)


I = Tasks
J = Units
S = States
U = [Steam]
WS = [Waste_1, Waste_2]
T = range(0, Cycle_time + 1)

# I = [i.name for i in Tasks]
# J = [i.name for i in Units]
# S = [i.name for i in States]
# U = [Steam.name]
# WS = [Waste_1.name, Waste_2.name]
# T = range(0, Cycle_time + 1)

TK = [Milk_Silo, Culture_Silo, Curd_Tank_1, Curd_Tank_2, Curd_Tank_3, Waste_Tank_1, Waste_Tank_2, Cream_Silo, Warehouse_9, Warehouse_10, Protein_Tank, Lactose_Tank]


# Ij = {j: j.__dict__['tasks'] for j in Units if j.__dict__['tasks'] is not None}
Ki = {i: i.__dict__['units'] for i in Tasks}                                    # K[i] set of units capable of task i

Ij = {}
for k,v in Ki.items():
    for x in v:
        Ij.setdefault(x,[]).append(k)

Si = {i: i.__dict__['input'] for i in I}
Sbari = {i: i.__dict__['output'] for i in I}
Ts = {}
for k,v in Si.items():
    for x in v:
         Ts.setdefault(x,[]).append(k)
Tbars = {}
for k,v in Sbari.items():
    for x in v:
         Tbars.setdefault(x,[]).append(k)
# print(Tbars, Ts)
Ks = {s: s.__dict__['storage'] for s in S if s.__dict__['storage'] is not None}

# print(Ks)
P = {(i,s): i.__dict__['output_time'][s] for i in I for s in i.__dict__['output']}          # P[(i,s)] time for task i output to state s  
pi = {i: max([P[(i,s)] for s in Sbari[i]]) for i in I}           # p[i] completion time for task i
print(P)
rho = {(i,s): i.__dict__['input_fraction'][s] for i in Tasks for s in i.__dict__['input']}        # rho[(i,s)] input fraction of task i from state s
rho_ = {(i,s): i.__dict__['output_fraction'][s] for i in Tasks for s in i.__dict__['output']}       # rho_[(i,s)] output fraction of task i to state s

print(rho, rho_)

model = ConcreteModel()

#Sets
model.I = Set(initialize = [i.name for i in I])
model.J = Set(initialize = [i.name for i in J])
model.S = Set(initialize = [i.name for i in S])
model.U = Set(initialize = [i.name for i in U])
model.WS = Set(initialize = [i.name for i in WS])
model.T = Set(initialize = T)
model.TK = Set(initialize = [i.name for i in TK])

# model.I = Set(initialize = I)
# model.J = Set(initialize = J)
# model.S = Set(initialize = S)
# model.U = Set(initialize = U)
# model.WS = Set(initialize = WS)
# model.T = Set(initialize = T)
# model.TK = Set(initialize = TK)


#Parameters
Ds = 137700/2
Ss0 = {s: s.__dict__['initial_value'] for s in S}

# #Variables
# model.Ej = Var(model.J, domain = Binary)
# model.Etk = Var(model.TK, domain = Binary)
# model.W = Var(model.I, model.J, model.T, domain = Binary)

model.Ej = Var(model.J, domain = NonNegativeReals)
model.Etk = Var(model.TK, domain = NonNegativeReals)
model.W = Var(model.I, model.J, model.T, domain = NonNegativeReals)

model.Vj = Var(model.J, domain = NonNegativeReals)
model.Vtk = Var(model.TK, domain = NonNegativeReals)
model.B = Var(model.I, model.J, model.T, domain = NonNegativeReals)
model.St = Var(model.S, model.T, domain = NonNegativeReals)
model.Waste = Var(model.WS, model.T, domain = NonNegativeReals)
model.Util = Var(model.U, model.T, domain = NonNegativeReals)
model.setup_cost = Var(domain = NonNegativeReals)
model.revenue = Var(domain = NonNegativeReals)

initial = [Culture.name, SkimMilk.name, Cream.name]

for s in model.S:
    if s not in initial:
        # print(s)
        model.St[s, 0].fix(0)
        
for s in model.S:
    if s not in [i.name for i in Ks]:     
        model.St[s,:].fix(0) 


# for s in initial:
#     model.St[s,0].fix(10000)


# from itertools import product 
# for i in J:
#     model.Ej[i].fix(0.2) 

# for i in TK:
#     model.Etk[i].fix(0.2)
    
# for i,j,k in product(I, J, T):
#     model.W[i,j,k].fix(0.2)


#Constraints

def unit_allocation(instance, unit, time):
    """Unit Allocation Constraints
    A unit can only be allocated to one task 
    Args:
        instance (_type_): _description_
        time (_type_): _description_
    """ 
    lhs = 0
    if unit in [i.name for i in Ij]:
        for i in Ij[unit]:
            if time  - pi[i] + 1 > 0:
                tdprime = time - pi[i] + 1
            else:
                tdprime = time - pi[i] + 1 + Cycle_time 
            for tprime in range(tdprime, time + 1):          
                lhs += instance.W[i,unit,tprime]
        expr = lhs <= instance.Ej[unit]
    else:
        expr = Constraint.Skip
    return expr


#%%
model.unit_allocation_cons = Constraint(model.J, model.T, rule = unit_allocation, doc = 'Unit Allocation Constraints')
#%%
def unit_capacity_lb(instance, task, unit):
    """Unit Capacity Lower Bound Constraints
        
    Args:
        instance (_type_): _description_
        time (_type_): _description_
    """ 
    if unit in Ki[task]:
        Vjmin = unit.__dict__['min_capacity']
        expr = Vjmin*instance.Ej[unit] <= instance.Vj[unit]
    else:
        expr = Constraint.Skip
    return expr

model.unit_capacity_lb_cons = Constraint(model.I, model.J, rule = unit_capacity_lb, doc = 'Unit Capacity LB Constraints')

def unit_capacity_ub(instance, task, unit):
    """Unit Capacity Upper Bound Constraints
        
    Args:
        instance (_type_): _description_
        time (_type_): _description_
    """ 
    if unit in Ki[task]:
        Vjmax = unit.__dict__['max_capacity']
        expr = instance.Vj[unit] <= Vjmax*instance.Ej[unit]
    else:
        expr = Constraint.Skip
    return expr

model.unit_capacity_ub_cons = Constraint(model.I, model.J, rule = unit_capacity_ub, doc = 'Unit Capacity UB Constraints')

def batch_size_lb(instance, task, unit, time):
    """Batch Size Production Lower Bound

    Args:
        instance (_type_): _description_
        task (_type_): _description_
        unit (_type_): _description_
        time (_type_): _description_
    """
    if unit in Ki[task]:
         Vjmin = unit.__dict__['min_capacity']
         expr = instance.W[task, unit, time]*Vjmin <= instance.B[task, unit, time]
    else:
        expr = Constraint.Skip
    return expr

model.batch_size_lb_cons = Constraint(model.I, model.J, model.T, rule = batch_size_lb, doc = 'Batch Size LB Constraints')

def batch_size_ub(instance, task, unit, time):
    """Batch Size Production Upper Bound

    Args:
        instance (_type_): _description_
        task (_type_): _description_
        unit (_type_): _description_
        time (_type_): _description_
    """
    if unit in Ki[task]:
         Vjmax = unit.__dict__['max_capacity']
         expr = instance.W[task, unit, time]*Vjmax <= instance.B[task, unit, time]
    else:
        expr = Constraint.Skip
    return expr

model.batch_size_ub_cons = Constraint(model.I, model.J, model.T, rule = batch_size_ub, doc = 'Batch Size UB Constraints')

def production_capacity(instance, task, unit, time):
    """Production constraint due to Unit Capacity

    Args:
        instance (_type_): _description_
        task (_type_): _description_
        unit (_type_): _description_
        time (_type_): _description_
    """
    if unit in Ki[task]:
        expr = instance.B[task, unit, time] <= instance.Vj[unit]
    else:
        expr = Constraint.Skip
    return expr

model.production_capacity_cons = Constraint(model.I, model.J, model.T, rule = production_capacity, doc = 'Production Capacity Constraints')

def storage_capacity_lb(instance, state, tanks, time):
    """Storage Upper Bound Constraints due to unit capacity

    Args:
        instance (_type_): _description_
        state (_type_): _description_
        tanks (_type_): _description_
        time (_type_): _description_
    """
    if state in Ks and tanks in Ks[state]:
        expr = 0 <= instance.St[state, time]
    else:
        expr = Constraint.Skip
    return expr

model.storage_capacity_lb_cons = Constraint(model.S, model.TK, model.T, rule = storage_capacity_lb, doc = 'Storage Capacity Upper Bound Constraints')

def storage_capacity_ub(instance, state, tanks, time):
    """Storage Upper Bound Constraints due to unit capacity

    Args:
        instance (_type_): _description_
        state (_type_): _description_
        tanks (_type_): _description_
        time (_type_): _description_
    """
    if state in Ks and tanks in Ks[state]:
        expr = instance.St[state, time] <= instance.Vtk[tanks]
    else:
        expr = Constraint.Skip
    return expr

model.storage_capacity_ub_cons = Constraint(model.S, model.TK, model.T, rule = storage_capacity_ub, doc = 'Storage Capacity Upper Bound Constraints')

def storage_lb(instance, state, tank):
    """Storage Lower Bound Constraints
        
    Args:
        instance (_type_): _description_
        time (_type_): _description_
    """ 
    if state in Ks and tank in Ks[state]:
        Vtkmin = tank.__dict__['min_capacity']
        expr = Vtkmin*instance.Etk[tank] <= instance.Vtk[tank]
    else:
        expr = Constraint.Skip
    return expr

model.storage_lb_cons = Constraint(model.S, model.TK, rule = storage_lb, doc = 'Storage LB Constraints')

def storage_ub(instance, state, tank):
    """Storage Upper Bound Constraints
        
    Args:
        instance (_type_): _description_
        time (_type_): _description_
    """ 
    if state in Ks and tank in Ks[state]:
        Vtkmax = tank.__dict__['max_capacity']
        expr = instance.Vtk[tank] <= Vtkmax*instance.Etk[tank]
    else:
        expr = Constraint.Skip
    return expr

model.storage_ub_cons = Constraint(model.S, model.TK, rule = storage_ub, doc = 'Storage UB Constraints')


def mass_balance(instance, state, time):
    """Mass Balance Constraints

    Args:
        instance (_type_): _description_
        state (_type_): _description_
        time (_type_): _description_
    """
    lhs = 0
    rhs = 0
    lhs = instance.St[state, time]
    if time != 0:
        rhs = instance.St[state, time -1] 
    if state in Ts:
        rhs -= sum(rho[task, state]*instance.B[task, unit, time] for task in Ts[state] for unit in Ki[task])
    if state in Tbars:
        for task in Tbars[state]:
            if time - P[task, state] >= 1:         
                rhs += sum(rho_[task, state]*instance.B[task, unit, time - P[task, state]] for unit in Ki[task]) 
            else:
                rhs += sum(rho_[task, state]*instance.B[task, unit, time - P[task, state] + Cycle_time] for unit in Ki[task])
    expr = lhs == rhs
    return expr


model.mass_balance_cons = Constraint(model.S, model.T, rule = mass_balance, doc = 'Mass Balance Constraints')

def production_requirement(instance, state):
    """Production Requirement Constraints

    Args:
        instance (_type_): _description_
    """
    if state is StdBld9 or state is StdBld10:
        expr = (instance.St[state, Cycle_time] - instance.St[state, 0])*Horizon <= Ds*Cycle_time
    else:
        expr = Constraint.Skip
    return expr

model.production_requirement_cons = Constraint(model.S, rule = production_requirement, doc = 'Production Requirement Constraints')

def utility(instance, utility, time):
    """Utility Constraints

    Args:
        instance (_type_): _description_
        utility (_type_): _description_
        time (_type_): _description_
    """
    return instance.Util[utility, time] == sum(alpha*instance.B[task, unit, time] for task in I for unit in Ki[task])

model.utility_cons = Constraint(model.U, model.T, rule = utility, doc = 'Utility Constraints')

def waste(instance, waste, time):
    """Waste Constraint

    Args:
        instance (_type_): _description_
        waste (_type_): _description_
        time (_type_): _description_
    """
    
    return instance.Waste[waste, time] == sum(beta*instance.B[task, unit, time] for task in I for unit in Ki[task])

model.waste_cons = Constraint(model.WS, model.T, rule = waste, doc = 'Waste Constraints')

def facility_cost(instance):
    """Cost of Facility

    Args:
        instance (_type_): _description_
    """
    
    return instance.setup_cost == (sum(instance.Ej[unit]*unit.__dict__['fixed_cost'] + instance.Vj[unit]*unit.__dict__['variable_cost'] for unit in instance.J) + sum(instance.Etk[tank]*tank.__dict__['fixed_cost'] + instance.Vtk[tank]*tank.__dict__['variable_cost'] for tank in instance.TK))*If*CCF 

model.facility_cost_cons = Constraint(rule = facility_cost, doc = 'Facility Cost Constraint')

def revenue_cost(instance):
    """Revenue

    Args:
        instance (_type_): _description_
    """
    return instance.revenue == (sum(state.__dict__['cost'] * (instance.St[state, Cycle_time] - instance.St[state, 0]) for state in instance.S) - sum(utility.__dict__['cost'] * instance.Util[utility, time] for utility in instance.U for time in instance.T))*Horizon/Cycle_time 

model.revenue_cost_cons = Constraint(rule = revenue_cost, doc = 'Revenue Constraint')

def objective(instance):
    """Objective Function to minimize annual cost

    Args:
        instance (_type_): _description_
    """
    return instance.revenue - instance.setup_cost

model.dual = Suffix(direction=Suffix.IMPORT_EXPORT)

        
model.Objective = Objective(rule = objective, sense = maximize, doc = 'Objective Function')

opt = SolverFactory('gurobi', solver_io = 'python')
results = opt.solve(model)
results.write()
#%%

model.pprint()
# def mass_balance(instance, state, time):
#     """Mass Balance Constraints

#     Args:
#         instance (_type_): _description_
#         state (_type_): _description_
#         time (_type_): _description_
#     """
#     print(state,time)
#     if state in Tbars:
#         # tprime = {}
#         # for i in Tbars[state]:
#         #     tprime.setdefault((i,state)).append(0)
#         #     # tprime[i].append(0)
#         #     if time - P[i, state] >= 1:
#         #         tprime[i, state] = time - P[i, state]
#         #     else:
#         #         tprime[i, state] = time - P[i, state] + Cycle_time
#         if time - P[task, state] >= 1:         
#             lhs = instance.St[state, time]
#             rhs = instance.St[state, time -1] - sum(rho(task, state)*instance.B[task, unit, time] for task in Ts for unit in Ki[task]) \
#                 + sum(rho_[task, state]*instance.B[task, unit, time - P[task, state]] for task in Tbars[state] for unit in Ki[task])
#             expr = lhs == rhs 
#         else:
#             lhs = instance.St[state, time]
#             rhs = instance.St[state, time -1] - sum(rho(task, state)*instance.B[task, unit, time] for task in Ts for unit in Ki[task]) \
#                 + sum(rho_[task, state]*instance.B[task, unit, time - P[task, state] + Cycle_time] for task in Tbars[state] for unit in Ki[task])
#             expr = lhs == rhs
#     else:
#         expr = Constraint.Skip
#     return expr
# %%
#Storage Inventory
pd.DataFrame([[model.St[s,t]() for s in model.S] for t in model.T], columns = model.S, index = model.T)

# %%
plt.figure(figsize=(10,10))
for (s,idx) in zip(model.S,range(0,len(model.S))):
    if s in Ks:
        plt.subplot(ceil(len(model.S)/3),3,idx+1)
        tlast,ylast = 0, s.__dict__['initial_value']
        for (t,y) in zip(list(model.T),[model.St[s,t]() for t in model.T]):
            plt.plot([tlast,t,t],[ylast,ylast,y],'b')
            #plt.plot([tlast,t],[ylast,y],'b.',ms=10)
            tlast,ylast = t,y
        plt.ylim(0,1.1*max(tank.__dict__['max_capacity'] for tank in Ks[s] ))
        plt.plot([0,Cycle_time],[max(tank.__dict__['max_capacity'] for tank in Ks[s] if s in Ks),max(tank.__dict__['max_capacity'] for tank in Ks[s] if s in Ks)],'r--')
        plt.title(s.__dict__['name'])
plt.tight_layout()
# %%
